# Домашняя работа 2. Как на самом деле решают МНК

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 2 — Эмпирический риск и метод наименьших квадратов |
| Опора | материал семинара 2 и лекций до него |
| Ожидаемое время | 3–4 часа |

На занятии мы вызывали `np.linalg.lstsq` и `LinearRegression`, не заглядывая внутрь. Дома разберёмся, что там происходит: проверим формулы матричного дифференцирования численно, реализуем пять способов решить одну и ту же задачу и увидим, где четыре из них ломаются. В конце — связь между предположением о шуме и выбором функции потерь.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=2)
describe_variant(variant)

---
# Задача 1. Матричное дифференцирование: проверка конечными разностями

В §6 лекции 1 выведены два тождества:

$$
\nabla_w (a^{\mathsf T} w) = a,
\qquad
\nabla_w (w^{\mathsf T} A w) = (A + A^{\mathsf T}) w,
$$

и из них — градиент квадрата нормы невязки
$\nabla_\theta\|X\theta - y\|^2 = 2X^{\mathsf T}(X\theta - y)$.

Проверить формулу можно, не доказывая её заново: сравнить с центральной
конечной разностью

$$
\frac{\partial f}{\partial w_k} \approx \frac{f(w + he_k) - f(w - he_k)}{2h}.
$$

Приём стоит запомнить: при реализации любого градиента «с нуля» (а это будет
в занятиях 3 и 4) такая проверка за пять строк ловит ошибку в выкладке.

### Задание 1.1. Функция `numeric_grad`

In [ ]:
def numeric_grad(f, w, h=1e-6):
    """Градиент центральными разностями.

    Для каждой координаты k: сдвинуть w на +h и -h по этой координате,
    взять разность значенийف и поделить на 2h.
    """
    # TODO
    raise NotImplementedError

### Задание 1.2. Проверка трёх формул

Проверьте все три тождества. Матрицу $A$ берите **несимметричной**: для
симметричной формула вырождается в $2Aw$, и ошибку «забыл транспонировать»
не видно.

In [ ]:
n = 6
w0 = rng.normal(size=n)
a = rng.normal(size=n)
A = rng.normal(size=(n, n))                # намеренно несимметричная
X_chk = rng.normal(size=(40, n))
y_chk = rng.normal(size=40)

# TODO: проверьте три формулы, сравнив аналитический градиент с численным:
#   f(w) = a^T w          -> a
#   f(w) = w^T A w        -> (A + A^T) w
#   f(w) = ||X w - y||^2  -> 2 X^T (X w - y)

### Задание 1.3. Выбор шага $h$

Постройте зависимость ошибки проверки от $h$ в двойном логарифмическом масштабе
для $h$ от $10^{-1}$ до $10^{-12}$. У кривой должен быть минимум — найдите его
и объясните обе ветви.

In [ ]:
# TODO: для h из np.logspace(-1, -12, 40) посчитайте максимальную ошибку
#       численного градиента для f(w) = ||Xw - y||^2 и постройте график
#       в двойном логарифмическом масштабе. Отметьте минимум.

> **Вывод.** Почему ошибка растёт и при слишком большом, и при слишком малом $h$? Где примерно оптимум и почему именно там?
>
> *(ваш ответ здесь)*

---
# Задача 2. Пять способов решить МНК

На занятии мы пользовались двумя (`solve` и `lstsq`) и мельком отметили, что
явное обращение матрицы — плохая идея. Теперь измерим, насколько плохая.

Реализуйте пять решателей одной и той же задачи $\min_\theta\|X\theta - y\|^2$:

1. `fit_inv` — буквально по формуле теоремы 1.20, через `np.linalg.inv`;
2. `fit_solve` — решение нормальных уравнений без обращения;
3. `fit_qr` — через QR-разложение: $X = QR$, затем $R\theta = Q^{\mathsf T}y$;
4. `fit_lstsq` — `np.linalg.lstsq`;
5. `fit_svd` — SVD вручную: $\theta = V\Sigma^{+}U^{\mathsf T}y$ с отсечением
   малых сингулярных чисел.

> **Напоминание — QR- и сингулярное разложение.** **QR:** любую матрицу $X$ размера $\ell\times p$ можно записать как $X = QR$,
> где $Q$ ортогональна ($Q^{\mathsf T}Q = I$), а $R$ верхнетреугольная. Тогда
> $\|X\theta - y\| = \|R\theta - Q^{\mathsf T}y\|$ (ортогональная матрица не
> меняет длину), и остаётся решить треугольную систему — без построения
> $X^{\mathsf T}X$. В NumPy: `np.linalg.qr`.
>
> **SVD (сингулярное разложение):** $X = U\Sigma V^{\mathsf T}$, где $U$ и $V$
> ортогональны, а $\Sigma$ диагональна с неотрицательными числами
> $\sigma_1\ge\dots\ge\sigma_p\ge0$ — *сингулярными числами*. Геометрически:
> любое линейное отображение — это поворот, растяжение по осям и ещё поворот.
> Псевдообратная $\Sigma^{+}$ получается заменой $\sigma_j$ на $1/\sigma_j$,
> причём слишком малые $\sigma_j$ **обнуляют**, а не обращают — иначе шум
> делится на почти ноль. В NumPy: `np.linalg.svd`. Строгая формулировка — теорема
> 8.9 лекции 8, там же SVD станет самостоятельным сюжетом.

In [ ]:
from scipy import linalg


def fit_inv(X, y):
    """(X^T X)^{-1} X^T y -- буквально по формуле теоремы 1.20."""
    raise NotImplementedError


def fit_solve(X, y):
    """Решение нормальных уравнений X^T X theta = X^T y без обращения."""
    raise NotImplementedError


def fit_qr(X, y):
    """X = QR (linalg.qr, mode='economic'), затем solve_triangular(R, Q^T y)."""
    raise NotImplementedError


def fit_lstsq(X, y):
    raise NotImplementedError


def fit_svd(X, y, tol=1e-12):
    """theta = V Sigma^+ U^T y; сингулярные числа меньше tol * s.max() обнулить."""
    raise NotImplementedError


SOLVERS = {"inv": fit_inv, "solve": fit_solve, "QR": fit_qr,
           "lstsq": fit_lstsq, "SVD вручную": fit_svd}

### Задание 2.1. Сверка на примере из конспекта

Все пять способов обязаны дать $\theta^*\approx(2.201,\ 0.893,\ 0.114)$
на выборке из восьми кошек (пример 1.21).

In [ ]:
f1 = np.array([1, 2, 2, 3, 2, 3, 1, 4], dtype=float)
f2 = np.array([1, 2, 3, 5, 7, 10, 2, 8], dtype=float)
y_cats = np.array([3.0, 4.2, 4.5, 5.5, 4.8, 6.0, 3.4, 6.6])
X_cats = np.column_stack([np.ones(8), f1, f2])

# TODO: примените все пять решателей и сведите в таблицу вместе с
#       отличием от theta* = (2.201, 0.893, 0.114) из конспекта.

### Задание 2.2. Где они ломаются

Матрица Вандермонда $X_{ij} = t_i^{\,j-1}$ печально известна обусловленностью.
Возьмите $t$ равномерно на $[0,1]$, задайте **точный** ответ
$\theta_{\text{истина}}$ и постройте $y = X\theta_{\text{истина}}$ **без шума**:
тогда любая ошибка — чисто вычислительная.

Постройте график $\|\hat\theta - \theta_{\text{истина}}\|$ от числа столбцов
для всех пяти методов и рядом — рост $\mathrm{cond}(X)$ и $\mathrm{cond}(X)^2$.

In [ ]:
t = np.linspace(0, 1, 120)
degrees = range(2, 15)

# TODO: для каждого числа столбцов p:
#   X_v = np.vander(t, p, increasing=True); theta_true случайный;
#   y_v = X_v @ theta_true (БЕЗ шума);
#   решите всеми пятью способами, запомните ||theta_hat - theta_true||
#   и cond(X_v).
# TODO: два графика в логарифмическом масштабе: ошибки методов и
#       cond(X), cond(X)^2, уровень 1/eps.

> **Вывод.** С какого числа обусловленности `inv` и `solve` дают бессмысленный ответ? Почему QR и SVD держатся дольше?
>
> *(ваш ответ здесь)*

---
# Задача 3. Шум определяет функцию потерь

Утверждение 1.15 связывает ERM и максимум правдоподобия: если положить
$\mathcal L(a_\theta, x, y) = -\ln\varphi(x,y,\theta)$, принципы совпадают.
Для гауссовского шума (пример 1.22)

$$
-\ln\varphi = C + \frac{1}{2\sigma^2}(g(x,\theta) - y)^2
\;\Longrightarrow\;
\theta_{\mathrm{ML}} = \arg\min_\theta\|X\theta - y\|^2 .
$$

А если шум лапласовский, $p(\varepsilon)\propto e^{-|\varepsilon|/b}$, то
$-\ln\varphi = C + |g(x,\theta) - y|/b$ — и получается **абсолютная** потеря.

Проверим оба утверждения и посмотрим, что будет при выбросах.

In [ ]:
from scipy import optimize


def neg_loglik_gauss(theta, X, y, sigma=1.0):
    """-ln L для модели y = X theta + N(0, sigma^2), без не зависящих от theta слагаемых."""
    raise NotImplementedError


def neg_loglik_laplace(theta, X, y, b=1.0):
    """-ln L для модели y = X theta + Laplace(0, b)."""
    raise NotImplementedError


n = 120
x = np.sort(rng.uniform(0, 1, n))
X_lin = np.column_stack([np.ones(n), x])
THETA_TRUE = np.array([2.0, 3.0])
y_clean = X_lin @ THETA_TRUE + rng.normal(0, 0.3, n)

y_dirty = y_clean.copy()
outliers = rng.choice(n, size=6, replace=False)
y_dirty[outliers] += rng.choice([-1, 1], 6) * rng.uniform(6, 12, 6)

### Задание 3.1. ММП против МНК

Для чистых и для испорченных данных найдите $\theta$ тремя способами: МНК,
минимизацией гауссовского $-\ln L$ и минимизацией лапласовского.
Лапласовская функция негладкая — используйте `method="Nelder-Mead"`.

In [ ]:
# TODO: для y_clean и y_dirty найдите theta тремя способами
#       (lstsq, минимизация neg_loglik_gauss, минимизация neg_loglik_laplace)
#       и сведите в таблицу. Сравните с THETA_TRUE.

In [ ]:
# TODO: нарисуйте данные с выбросами, обе подогнанные прямые
#       (гауссовскую и лапласовскую) и истинную зависимость.

> **Вывод.** Совпали ли ММП при гауссовском шуме и МНК? Какая прямая устояла против выбросов и почему? Свяжите ответ с частью 1 семинара.
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Вы решаете МНК на данных, где один признак измерен в рублях, а другой — в долях единицы. Как это скажется на $\mathrm{cond}(X)$ и что стоит сделать до обучения?
2. В каком случае вы предпочтёте абсолютную потерю квадратичной, даже зная, что шум гауссовский?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.